# Table 2. Feeds, yields and yield coefficients on the 200 evaluation batches

Sugar, oil and PAA inputs and yield coefficients over the 200 evaluation batches.
Input: traj_b5v10.npz (pyIndPenSim replay on a 200-batch grid; override with TRAJ) for no
intervention, APC, PPO, the estimated-PAA BO controller and CEM.
Trajectories are interpolated from the grid to 0.2 h steps; discharge uses the recipe F_dis and is
checked by the ratio of reconstructed to reported yield ("chk"). Sugar kg = 0.6 x integral(F_s); Oil kg = 1.0 x integral(F_oil); PAA kg = PAA_c (per batch, ~520 g/L) x integral(F_PAA) Substrate consumed = sugar + oil - final S*V - discharged S Biomass formed = final X*V + discharged X - initial X*V Y_X/S, Y_P/S, Y_P/X, Y_P/PAA (kg penicillin / kg PAA), and sugar and PAA used per kg penicillin.
Yields come from each arm's evaluation yield files (with the divergence rule applied). Two BO batches
without trajectories (#86, #179; integration failure) are excluded from the coefficients only.

In [1]:
import sys; sys.path.insert(0, '..')          # repository root, for paths.py
from paths import DATA, OUTD                   # data/ and output/

In [2]:
import glob, numpy as np
G = DATA; OUT = OUTD + 'table2_cost'
tb = np.load(G + 'testset_fig5_REPORTONLY_random200.npz'); S = [int(s) for s in tb['seeds']]; LENS = tb['lens'].astype(int); U = tb['u']; H = 0.2; NB = 200
PAAC = tb['par'][:, 0, 74] / 1e6                                    # kg/L
import os
z = dict(np.load(G + os.environ.get('TRAJ', 'traj_b5v10.npz')))   # trajectories for all arms (BO: 10 repeated runs), read once
FR = z['frac']; CLS = z['cls']
got = {}
for f in sorted(glob.glob(G + ('cem_fine_w45_wide_b5_s10d12_s*.npz' if os.environ.get('CEM_SRC', 'w45') == 'w45' else 'cem_fine_w04_wide_b5_s*.npz'))):
    q = np.load(f, allow_pickle=True)
    for s, y in zip(q['seeds'], q['fine_yield']):
        if y is not None and np.isfinite(float(y)): got[int(s)] = float(y)
YLD = {'base': tb['ybase'][None].astype(float), 'apc': np.load(G + 'apc_streamA_b5_v2.npz', allow_pickle=True)['y_apcA'][None].astype(float),
       'rl': np.array([np.load(G + f'fin_faith_obs16_rand200_ts{i}_yield.npy') for i in range(10)]),
       'bo': (np.array([np.load(G + f'fin_boctrl_patch_s{i}_rand200_yield.npy') for i in range(10)]) if os.environ.get('BO_PATCH', '1') == '1' else np.load(G + 'fin_boctrl_rand200_yield.npy')[None]),
       'shcem': np.array([[got.get(s, np.nan) for s in S]])}
NAME = {'base': 'No intervention', 'apc': 'APC', 'bo': 'Estimated PAA + BO', 'rl': 'PPO', 'shcem': 'CEM' if os.environ.get('CEM_SRC', 'w45') == 'w45' else 'CEM'}
def calc(key):
    yl = YLD[key]; R_ = yl.shape[0]; M = {k: np.full((R_, NB), np.nan) for k in ('sugar', 'oil', 'paa', 'cons', 'bio', 'yld', 'chk')}
    for r in range(R_):
        for j in range(NB):
            g = lambda v: z[f'{key}_{v}'][r * NB + j] if z[f'{key}_{v}'].shape[0] > NB else z[f'{key}_{v}'][j]
            n = LENS[j]; t = FR * n * H; ts = np.arange(n) * H
            vals = [g(v) for v in ('X', 'S', 'Pconc', 'PV', 'Fs_u', 'Foil_u', 'Fpaa_u')]
            if not all(np.isfinite(v).all() for v in vals): continue
            X, Sx, P, PV, Fs, Fo, Fp = (np.interp(ts, t, v) for v in vals)
            V = np.where(P > 1, PV * 1000 / np.maximum(P, 1e-9), np.nan); ok = np.isfinite(V)
            if ok.sum() < 2: continue
            V = np.interp(ts, ts[ok], V[ok]); Fd = np.abs(U[j, :n, 12])
            sug = 0.6 * Fs.sum() * H; oil = 1.0 * Fo.sum() * H; paa = PAAC[j] * Fp.sum() * H
            rX, rS, rP = ((Fd * c).sum() * H / 1000 for c in (X, Sx, P))
            M['sugar'][r, j], M['oil'][r, j], M['paa'][r, j] = sug, oil, paa
            M['cons'][r, j] = sug + oil - Sx[-1] * V[-1] / 1000 - rS
            M['bio'][r, j] = X[-1] * V[-1] / 1000 + rX - X[0] * V[0] / 1000
            M['yld'][r, j] = yl[r, j]; M['chk'][r, j] = (PV[-1] + rP) / yl[r, j]
    return {k: np.nanmean(v, 0) for k, v in M.items()}
R = {k: calc(k) for k in NAME}
L = ['200 evaluation batches · mean per batch (kg) · discharge check ' + ' · '.join(f'{k} {np.nanmean(R[k]["chk"]):.3f}' for k in NAME), '',
     f'{"":20s}{"Yield":>7s}{"Sugar":>8s}{"Oil":>7s}{"PAA":>6s}{"Subst.":>8s}{"Biom.":>7s}{"Y_X/S":>7s}{"Y_P/S":>7s}{"Y_P/X":>7s}{"Y_P/PAA":>8s}{"S/P":>6s}{"PAA/P":>7s}']
for k, nm in NAME.items():
    d = R[k]; f = lambda a, b: np.nanmean(d[a] / d[b])
    L.append(f'{nm:20s}{np.nanmean(d["yld"]):7.0f}{np.nanmean(d["sugar"]):8.0f}{np.nanmean(d["oil"]):7.0f}{np.nanmean(d["paa"]):6.0f}'
             f'{np.nanmean(d["cons"]):8.0f}{np.nanmean(d["bio"]):7.0f}{f("bio", "cons"):7.3f}{f("yld", "cons"):7.3f}{f("yld", "bio"):7.3f}'
             f'{f("yld", "paa"):8.2f}{f("sugar", "yld"):6.2f}{f("paa", "yld"):7.3f}')
rng = np.random.default_rng(0)
def ci(d): d = d[np.isfinite(d)]; b = d[rng.integers(0, len(d), (5000, len(d)))].mean(1); return d.mean(), *np.percentile(b, [2.5, 97.5])
L.append('')
for a, b in (('rl', 'apc'), ('bo', 'apc'), ('rl', 'bo')):
    for nm, fn in (('Sugar kg', lambda d: d['sugar']), ('PAA kg', lambda d: d['paa']), ('Y_X/S', lambda d: d['bio'] / d['cons']),
                   ('Y_P/S', lambda d: d['yld'] / d['cons']), ('Y_P/PAA', lambda d: d['yld'] / d['paa']), ('Sugar/P', lambda d: d['sugar'] / d['yld'])):
        dd = fn(R[a]) - fn(R[b])
        L.append(f'  {NAME[a]} − {NAME[b]} · {nm}: %+.3f [%+.3f, %+.3f]' % ci(dd) + '  by class ' + ' '.join(f'{np.nanmean(dd[CLS == c]):+.3f}' for c in range(3)))
txt = '\n'.join(L); print(txt); open(OUT + '.txt', 'w').write(txt)
np.savez(OUT + '.npz', cls=CLS, **{f'{k}|{m}': v for k in NAME for m, v in R[k].items()})

200 evaluation batches · mean per batch (kg) · discharge check base 1.000 · apc 0.999 · bo 1.000 · rl 0.999 · shcem 0.997

                      Yield   Sugar    Oil   PAA  Subst.  Biom.  Y_X/S  Y_P/S  Y_P/X Y_P/PAA   S/P  PAA/P
No intervention        2800   11033   6010  1020   14843   2709  0.185  0.185  1.017    2.85  4.32  0.411
APC                    3467   11033   6010   840   17043   2928  0.173  0.204  1.184    4.12  3.18  0.243
Estimated PAA + BO     3650   13144   6010   853   18992   3258  0.173  0.194  1.125    4.28  3.61  0.235
PPO                    3708   13831   6015   886   19521   3343  0.173  0.191  1.112    4.20  3.75  0.241
CEM                    4114   14939   6108   889   21047   3537  0.169  0.196  1.162    4.63  3.64  0.216

  PPO − APC · Sugar kg: +2798.038 [+2667.558, +2926.622]  by class +2986.684 +2573.806 +2991.508
  PPO − APC · PAA kg: +46.835 [+37.082, +56.646]  by class +94.375 +44.914 +0.837
  PPO − APC · Y_X/S: -0.000 [-0.001, +0.000]  by class -0.001

## statistics of the efficiency metrics.

Uses the per-batch values written by table2_cost.py (table2_cost.npz).
Metrics: Y_X/S, Y_P/S, Y_P/PAA, sugar (kg) and PAA (kg) per kg penicillin.
Tests: Friedman test across the 5 arms with batches as blocks; pairwise Wilcoxon signed-rank tests
(PPO-APC, BO-APC, PPO-BO, CEM-PPO) with Holm correction within each metric; mean difference with a
batch-bootstrap 95 % CI; effect size dz.

In [3]:
import numpy as np
from scipy import stats
R = DATA; z = np.load(OUTD + 'table2_cost.npz')
ARMS = ['base', 'apc', 'bo', 'rl', 'shcem']; NM = {'base': 'None', 'apc': 'APC', 'bo': 'BO', 'rl': 'PPO', 'shcem': 'CEM'}
MET = {'Y_X/S': lambda k: z[f'{k}|bio'] / z[f'{k}|cons'], 'Y_P/S': lambda k: z[f'{k}|yld'] / z[f'{k}|cons'],
       'Y_P/PAA': lambda k: z[f'{k}|yld'] / z[f'{k}|paa'], 'Sugar per P': lambda k: z[f'{k}|sugar'] / z[f'{k}|yld'],
       'PAA per P': lambda k: z[f'{k}|paa'] / z[f'{k}|yld']}
PAIRS = [('rl', 'apc'), ('bo', 'apc'), ('rl', 'bo'), ('shcem', 'rl')]
rng = np.random.default_rng(0); L = []
def holm(ps):
    o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
    for r, i in enumerate(o): run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1, run)
    return adj
for name, f in MET.items():
    V = {k: f(k) for k in ARMS}; ok = np.all([np.isfinite(V[k]) for k in ARMS], 0)
    fr = stats.friedmanchisquare(*[V[k][ok] for k in ARMS]).pvalue
    L.append(f'{name:12s} n={ok.sum()}  mean ' + ' · '.join(f'{NM[k]} {V[k][ok].mean():.4f}' for k in ARMS) + f'   Friedman p={fr:.1e}')
    ps, rows = [], []
    for a, b in PAIRS:
        d = V[a][ok] - V[b][ok]; bs = d[rng.integers(0, len(d), (10000, len(d)))].mean(1)
        ps.append(stats.wilcoxon(d).pvalue); rows.append((a, b, d.mean(), *np.percentile(bs, [2.5, 97.5]), d.mean() / d.std(ddof=1), (d > 0).mean()))
    for (a, b, m, lo, hi, dz, pos), p in zip(rows, holm(np.array(ps))):
        L.append(f'    {NM[a]}−{NM[b]}: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  Holm p={p:.1e}  dz={dz:+.2f}  positive {pos*100:.0f}%')
txt = '\n'.join(L); print(txt); open(OUTD + 'table2_stats.txt', 'w').write(txt)

Y_X/S        n=200  mean None 0.1848 · APC 0.1729 · BO 0.1727 · PPO 0.1725 · CEM 0.1694   Friedman p=2.2e-67
    PPO−APC: -0.0004 [-0.0010, +0.0003]  Holm p=1.4e-04  dz=-0.08  positive 34%
    BO−APC: -0.0002 [-0.0007, +0.0005]  Holm p=1.4e-04  dz=-0.04  positive 37%
    PPO−BO: -0.0002 [-0.0006, +0.0002]  Holm p=6.4e-07  dz=-0.08  positive 30%
    CEM−PPO: -0.0031 [-0.0037, -0.0025]  Holm p=8.3e-22  dz=-0.67  positive 14%
Y_P/S        n=200  mean None 0.1854 · APC 0.2041 · BO 0.1936 · PPO 0.1911 · CEM 0.1964   Friedman p=2.3e-73
    PPO−APC: -0.0130 [-0.0144, -0.0116]  Holm p=8.8e-34  dz=-1.27  positive 2%
    BO−APC: -0.0104 [-0.0117, -0.0092]  Holm p=8.8e-34  dz=-1.14  positive 4%
    PPO−BO: -0.0026 [-0.0036, -0.0016]  Holm p=1.1e-10  dz=-0.35  positive 21%
    CEM−PPO: +0.0053 [+0.0035, +0.0073]  Holm p=1.6e-05  dz=+0.39  positive 61%
Y_P/PAA      n=200  mean None 2.8527 · APC 4.1238 · BO 4.2784 · PPO 4.2011 · CEM 4.6306   Friedman p=1.4e-116
    PPO−APC: +0.0773 [+0.0194, +0.1328

2136